# Downscaled `snw` exploration

This notebook is for exploring downscaled snow water equivalent (snw) data. It was designed to work with the zarr outputs from the 4km ERA5-based CMIP6 downscaling effort.

In [ ]:
# Set BASE_DIR environment variable to your path
# export BASE_DIR=/path/to/cmip6_4km_downscaling

# Pass parameters via command line arguments like so, for example:
# papermill downscaled_snw.ipynb output.ipynb -p models 'EC-Earth3-Veg' -p scenarios 'historical ssp370 ssp585' -p threshold 1524
models = "EC-Earth3-Veg"
scenarios = "historical ssp370 ssp585"
threshold = 1524

In [ ]:
# Parameters
models = "EC-Earth3-Veg"
scenarios = "historical ssp126 ssp370 ssp585"
threshold = 1524

In [ ]:
import math
import os
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
from pathlib import Path
from xclim.core.units import convert_units_to
import matplotlib.pyplot as plt
import gc
from baeda import projected_coords
from IPython.display import Markdown

import warnings
warnings.filterwarnings('ignore')

models = models.split(" ")
scenarios = scenarios.split(" ")

base_dir = Path(
    os.getenv("BASE_DIR", "/beegfs/CMIP6/crstephenson/EC-Earth3-Veg/cmip6_4km_downscaling")
)

ref_dir = base_dir.joinpath("era5_zarr")
cmip6_dir = base_dir.joinpath("cmip6_zarr")
adj_dir = base_dir.joinpath("adjusted")

random_points = []

In [ ]:
def open_snw(model, scenario):
    zarr_store = adj_dir.joinpath(f"snw_{model}_{scenario}_adjusted.zarr")
    ds = xr.open_zarr(zarr_store)
    snw = convert_units_to(ds.snw, "mm")
    return snw


def get_random_points(base, snw_historical, n=5):
    if len(random_points) >= n:
        return random_points
    for _ in range(n):
        snw_historical_extr = None
        while snw_historical_extr is None or np.isnan(snw_historical_extr).all():
            random_x = np.random.choice(base.x.values)
            random_y = np.random.choice(base.y.values)
            snw_historical_extr = snw_historical.sel(x=random_x, y=random_y)
        random_points.append((random_x, random_y))
    return random_points


def plot_adj_snw(model, scenario, snw):
    """Plot maps and histograms of adjusted (downscaled) data highlighting high snw pixels."""
    snw_mean = snw.mean("time")

    snw_high = snw > threshold
    snw_high_count = snw_high.sum("time")

    snw_high_values = snw.where(snw_high).values.flatten()
    snw_high_values = snw_high_values[~np.isnan(snw_high_values)]

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"High snw analysis for {model}, {scenario}", fontsize=14)

    axs[0].imshow(
        snw_mean.transpose("y", "x").values,
        cmap="Greys",
        alpha=0.5,
        interpolation="none",
    )

    masked_high = np.ma.masked_where(
        snw_high_count.transpose("y", "x") == 0, snw_high_count.transpose("y", "x")
    )
    im = axs[0].imshow(masked_high, cmap="Blues", alpha=0.8, interpolation="none")

    plt.colorbar(im, ax=axs[0], label=f"Count of days > {threshold}mm SWE (blue overlay)")
    axs[0].set_title(f"Mean snw (grey) with days > {threshold}mm (blue overlay)")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")

    if snw_high_values.size > 0:
        axs[1].hist(snw_high_values, bins=30, color="blue", alpha=0.7)
        axs[1].set_xlabel("snw (mm)")
        axs[1].set_ylabel("Frequency")
        axs[1].set_title(f"Histogram of daily snw values > {threshold}mm")
        percent_above_threshold = threshold * snw_high_values.size / snw.size
        axs[1].text(
            0.98,
            0.98,
            (
                f"{percent_above_threshold:.3f}% > {threshold}mm"
                if np.round(percent_above_threshold, 3) > 0
                else f"~0% > {threshold}mm"
            ),
            ha="right",
            va="top",
            transform=axs[1].transAxes,
            fontsize=12,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
        )
    else:
        axs[1].text(
            0.5,
            0.5,
            f"No snw values above {threshold}mm found.",
            ha="center",
            va="center",
            fontsize=12,
        )
        axs[1].set_axis_off()

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

    del snw
    gc.collect()


def get_top6_high_pixels(snw):
    snw = snw.transpose("y", "x", "time")
    snw_high_count = (snw > threshold).sum(dim="time").load()

    if np.sum(snw_high_count.values > 0) < 6:
        return []

    top6_yx = np.unravel_index(np.argsort(snw_high_count.values.ravel())[-6:], snw_high_count.shape)

    top6_x = snw_high_count.x.values[top6_yx[1]][::-1]
    top6_y = snw_high_count.y.values[top6_yx[0]][::-1]

    return list(zip(top6_x, top6_y))


def plot_top6_high_pixels(model, scenario, snw, top6_pixels):
    snw_mean = snw.mean("time")
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    snw_mean.transpose("y", "x").plot(ax=ax, cmap="Greys")
    top6_x, top6_y = zip(*top6_pixels)
    ax.scatter(top6_x, top6_y, color="blue", s=5, label=f"Top 6 high snw pixels (most occurrences of >{threshold}mm)")
    plt.title(f"Mean snw (grey) with top 6 pixels > {threshold}mm (blue) for {model}, {scenario}", pad=20)
    plt.legend()
    plt.show()


def plot_top6_high_ecdfs(
    era5_ds, hist_ds, sim_ds, snw_historical, snw, top6_pixels, thresh=None, names=None
):
    """Plot ECDFs for April data (peak snowpack month in Alaska)."""
    n = len(top6_pixels)
    ncols = math.ceil(n / 2)
    nrows = 2

    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), sharey=True)
    axs = axs.flatten()

    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    april_filter = lambda da: da.sel(time=da.time.dt.month == 4)

    for i, (x, y) in enumerate(top6_pixels):
        era5_extr = convert_units_to(
            era5_ds.snow_sum.sel(x=x, y=y, method="nearest").rename("snw"), "mm"
        )
        hist_extr = convert_units_to(
            hist_ds.snw.sel(x=x, y=y, method="nearest"), "mm"
        )
        sim_extr = convert_units_to(
            sim_ds.snw.sel(x=x, y=y, method="nearest"), "mm"
        )
        snw_historical_extr = snw_historical.sel(x=x, y=y, method="nearest").load()
        snw_extr = snw.sel(x=x, y=y, method="nearest").load()

        window_df = pd.concat(
            [
                april_filter(era5_extr)
                .assign_coords(source="ERA5")
                .to_dataframe()
                .reset_index(),
                april_filter(hist_extr)
                .rename("snw")
                .assign_coords(source=f"{model}_historical")
                .to_dataframe()
                .reset_index(),
                april_filter(sim_extr)
                .rename("snw")
                .assign_coords(source=f"{model}_{scenario}")
                .to_dataframe()
                .reset_index(),
                april_filter(snw_historical_extr)
                .assign_coords(source=f"{model}_historical_downscaled")
                .to_dataframe()
                .reset_index(),
                april_filter(snw_extr)
                .rename("snw")
                .assign_coords(source=f"{model}_{scenario}_downscaled")
                .to_dataframe()
                .reset_index(),
            ]
        )[["time", "source", "snw"]]

        window_df.reset_index(inplace=True)

        color_mapping = {
            "ERA5": "black",
            f"{model}_historical": "lightblue",
            f"{model}_{scenario}": "lightgreen",
            f"{model}_historical_downscaled": "blue",
            f"{model}_{scenario}_downscaled": "green",
        }
        sns.ecdfplot(
            data=window_df,
            x="snw",
            hue="source",
            ax=axs[i],
            palette=color_mapping,
        )
        if thresh:
            axs[i].set_xlim(left=thresh)
        axs[i].set_title(f"Pixel {i+1}\n(x={x:.0f}, y={y:.0f})" if names is None else names[i])
        axs[i].set_xlabel("snw (mm)")
        axs[i].set_ylabel("ECDF" if i % ncols == 0 else "")

    for j in range(i + 1, nrows * ncols):
        fig.delaxes(axs[j])

    if names is None:
        suptitle = (
            f"ECDF of snw for top 6 high snw pixels (most occurrences of >{threshold}mm) for {model}.\n"
            f"April data only (peak snowpack month in Alaska)"
        )
    else:
        suptitle = (
            f"ECDF of snw for 6 specified pixels for {model}.\n"
            f"April data only (peak snowpack month in Alaska)"
        )
    plt.suptitle(suptitle, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


def plot_doy_means(era5_ds, hist_ds, sim_ds, snw_historical, snw, x, y):
    sim_ds["snw"] = convert_units_to(sim_ds["snw"], "mm")
    hist_ds["snw"] = convert_units_to(hist_ds["snw"], "mm")

    sim_doy_mean = sim_ds["snw"].sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    hist_doy_mean = hist_ds["snw"].sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    ref_doy_mean = era5_ds["snow_sum"].sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    snw_historical_doy_mean = snw_historical.sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    snw_doy_mean = snw.sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")

    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    plt.figure(figsize=(9, 5))
    plt.plot(ref_doy_mean["dayofyear"], ref_doy_mean, label="ERA5", color="black")
    plt.plot(hist_doy_mean["dayofyear"], hist_doy_mean, label=f"{model} historical", color="lightblue")
    plt.plot(sim_doy_mean["dayofyear"], sim_doy_mean, label=f"{model} {scenario}", color="lightgreen")
    plt.plot(snw_historical_doy_mean["dayofyear"], snw_historical_doy_mean, label=f"{model} historical downscaled", color="blue")
    plt.plot(snw_doy_mean["dayofyear"], snw_doy_mean, label=f"{model} {scenario} downscaled", color="green")
    plt.xlabel("Day of Year")
    plt.ylabel("snw (mm)")
    plt.title(f"Day-of-year mean snw: {model}, {scenario}, pixel: (x={x}, y={y})")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_random_ecdf(base, era5_ds, hist_ds, sim_ds, snw_historical, snw, ixy, thresh=None):
    """Plot ECDF for April data at a random point location."""
    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    fig, axs = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle(
        f"snw analysis for downscaled {model}, {scenario} at point location",
        fontsize=14,
    )

    base.plot(ax=axs[0], cmap="Greys", alpha=0.5, add_colorbar=False, x="x")

    random_x = ixy[0]
    random_y = ixy[1]

    axs[0].scatter(random_x, random_y, color="blue", s=5, label="Random Pixel")

    april_filter = lambda da: da.sel(time=da.time.dt.month == 4)

    era5_extr = convert_units_to(
        era5_ds.snow_sum.sel(x=random_x, y=random_y).rename("snw"), "mm"
    )
    hist_extr = convert_units_to(hist_ds.snw.sel(x=random_x, y=random_y), "mm")
    sim_extr = convert_units_to(sim_ds.snw.sel(x=random_x, y=random_y), "mm")
    snw_historical_extr = snw_historical.sel(x=random_x, y=random_y).load()
    snw_extr = snw.sel(x=random_x, y=random_y).load()

    window_df = pd.concat(
        [
            april_filter(era5_extr)
            .assign_coords(source="ERA5")
            .to_dataframe()
            .reset_index(),
            april_filter(hist_extr)
            .rename("snw")
            .assign_coords(source=f"{model}_historical")
            .to_dataframe()
            .reset_index(),
            april_filter(sim_extr)
            .rename("snw")
            .assign_coords(source=f"{model}_{scenario}")
            .to_dataframe()
            .reset_index(),
            april_filter(snw_historical_extr)
            .assign_coords(source=f"{model}_historical_downscaled")
            .to_dataframe()
            .reset_index(),
            april_filter(snw_extr)
            .rename("snw")
            .assign_coords(source=f"{model}_{scenario}_downscaled")
            .to_dataframe()
            .reset_index(),
        ]
    )[["time", "source", "snw"]]

    window_df.reset_index(inplace=True)

    color_mapping = {
        "ERA5": "black",
        f"{model}_historical": "lightblue",
        f"{model}_{scenario}": "lightgreen",
        f"{model}_historical_downscaled": "blue",
        f"{model}_{scenario}_downscaled": "green",
    }

    sns.ecdfplot(
        data=window_df,
        x="snw",
        hue="source",
        ax=axs[1],
        palette=color_mapping,
    )
    if thresh:
        plt.xlim(left=thresh)
    axs[1].set_xlabel("snw (mm)")

    plt.tight_layout()
    plt.show()

    return window_df

In [ ]:
era5_ds = xr.open_dataset(ref_dir.joinpath("snow_sum_era5.zarr"))
era5_ds["snow_sum"] = convert_units_to(era5_ds.snow_sum, "mm")

### Choose hand-picked places

In [ ]:
places1 = ["Fairbanks", "Anchorage", "Nome", "Yakutat", "Utqiagvik", "near_McGrath"]
pixels1 = [(projected_coords[k]["x"], projected_coords[k]["y"]) for k in places1]

places2 = [
    "near_McGrath",
    "near_Arctic_Village",
    "warmest",
    "coldest",
    "wettest",
    "driest",
]
pixels2 = [(projected_coords[k]["x"], projected_coords[k]["y"]) for k in places2]

In [ ]:
for model in models:
    display(Markdown(f"# {model}"))
    hist_ds = xr.open_dataset(cmip6_dir.joinpath(f"snw_{model}_historical.zarr"))
    snw_historical = open_snw(model, "historical")

    era5_count = (era5_ds.snow_sum > threshold).sum("time").load().sum().item()
    historical_count = (convert_units_to(hist_ds.snw, "mm") > threshold).sum("time").load().sum().item()
    adj_historical_count = (snw_historical > threshold).sum("time").load().sum().item()

    for scenario in scenarios:
        display(Markdown(f"## {model}, {scenario}"))
        snw = open_snw(model, scenario)
        sim_ds = xr.open_dataset(cmip6_dir.joinpath(f"snw_{model}_{scenario}.zarr"))

        sim_count = (convert_units_to(sim_ds.snw, "mm") > threshold).sum("time").load().sum().item()
        adj_count = (snw > threshold).sum("time").load().sum().item()

        top6_pixels = get_top6_high_pixels(snw)
        if top6_pixels:
            display(Markdown(f"### High snw analysis"))
            plot_adj_snw(model, scenario, snw)

            print(f"Counts of all pixels > {threshold}mm SWE across all days:")
            print(f"ERA5: {era5_count:,} pixels")
            if scenario != "historical":
                print(f"{model} historical (unadjusted): {historical_count:,} pixels")
                print(f"{model} historical (downscaled): {adj_historical_count:,} pixels")
            print(f"{model} {scenario} (unadjusted): {sim_count:,} pixels")
            print(f"{model} {scenario} (downscaled): {adj_count:,} pixels")

            display(Markdown(f"### Top 6 pixels with highest counts of snw > {threshold}mm"))
            print(f"Top 6 (x, y) coordinates with highest counts of days > {threshold}mm:")
            for idx, (x, y) in enumerate(top6_pixels, start=1):
                print(
                    f"{idx}. (x: {x:.2f}, y: {y:.2f}), count above {threshold}mm: {(snw.sel(x=x, y=y).values > threshold).sum()}"
                )
            plot_top6_high_pixels(model, scenario, snw, top6_pixels)
            if scenario != "historical":
                plot_top6_high_ecdfs(era5_ds, hist_ds, sim_ds, snw_historical, snw, top6_pixels)

                display(Markdown(f"### Day-of-year (DOY) means"))
                for i in range(6):
                    plot_doy_means(
                        era5_ds, hist_ds, sim_ds, snw_historical, snw, x=top6_pixels[i][0], y=top6_pixels[i][1]
                    )
        else:
            print(f"Fewer than 6 pixels found with snw > {threshold}mm.")

        if scenario != "historical":
            display(Markdown(f"### Hand-picked places (April data)"))
            plot_top6_high_ecdfs(era5_ds, hist_ds, sim_ds, snw_historical, snw, pixels1, names=places1)
            plot_top6_high_ecdfs(era5_ds, hist_ds, sim_ds, snw_historical, snw, pixels2, names=places2)

            display(Markdown(f"### Random locations (April data)"))
            base = snw.max("time")
            random_points = get_random_points(base, snw_historical)
            for ixy in random_points:
                plot_random_ecdf(base, era5_ds, hist_ds, sim_ds, snw_historical, snw, ixy)